# Inside a model: what did we actually download?

This notebook looks inside the package behind a small language model. We will answer questions like these:

- How many **parameters** does the model have?
- How many weight arrays and matrices are stored?
- What are the **weights** themselves?
- What does it mean that the model has **open weights**?
- What files did WebLLM download into the browser cache?

A **parameter** is a number learned during training. A **weight** is a parameter that tells the model how strongly one value should affect another.

## Step 1: Get set up

This notebook does not run the model itself. Instead, it looks up public metadata from Hugging Face about the original model and the converted WebLLM package.

In [ ]:
import js
import math
import pandas as pd


async def fetch_json(url):
    response = await js.fetch(url)
    if not response.ok:
        raise RuntimeError(f"Could not load {url}. Status {response.status}.")
    return (await response.json()).to_py()


def format_int(value):
    return f"{int(value):,}"


def format_millions(value):
    return f"{value / 1_000_000:.1f} million"


def format_billions(value):
    return f"{value / 1_000_000_000:.2f} billion"


## Step 2: Pick a model family

We need two Hugging Face repositories for each example:

1. the **original model** from the model maker
2. the **converted model** from `mlc-ai`, which is the packaged WebLLM version

The default choice is Gemma 3 1B because it is one of the main teaching models in this project.

In [ ]:
MODEL_CHOICE = 1

MODEL_OPTIONS = [
    {
        'label': 'Gemma 3 1B Instruct',
        'original_repo': 'google/gemma-3-1b-it',
        'converted_repo': 'mlc-ai/gemma3-1b-it-q4f16_1-MLC',
    },
    {
        'label': 'Llama 3.2 1B Instruct',
        'original_repo': 'meta-llama/Llama-3.2-1B-Instruct',
        'converted_repo': 'mlc-ai/Llama-3.2-1B-Instruct-q4f16_1-MLC',
    },
    {
        'label': 'Qwen 2.5 0.5B Instruct',
        'original_repo': 'Qwen/Qwen2.5-0.5B-Instruct',
        'converted_repo': 'mlc-ai/Qwen2.5-0.5B-Instruct-q4f16_1-MLC',
    },
]

model_info = MODEL_OPTIONS[MODEL_CHOICE - 1]
print('Model family:   ' + model_info['label'])
print('Original repo:  ' + model_info['original_repo'])
print('WebLLM package: ' + model_info['converted_repo'])

## Step 3: Load public metadata

The next cell fetches:

- the original model card metadata
- the converted WebLLM package metadata
- the WebLLM chat config
- the WebLLM tensor inventory

The tensor inventory is especially useful because it lists the stored weight arrays, their shapes, and which binary shard file each one lives in.

In [ ]:
original_api = 'https://huggingface.co/api/models/' + model_info['original_repo']
converted_api = 'https://huggingface.co/api/models/' + model_info['converted_repo']
chat_config_url = 'https://huggingface.co/' + model_info['converted_repo'] + '/resolve/main/mlc-chat-config.json'
tensor_cache_url = 'https://huggingface.co/' + model_info['converted_repo'] + '/resolve/main/tensor-cache.json'

original_info = await fetch_json(original_api)
converted_info = await fetch_json(converted_api)
chat_config = await fetch_json(chat_config_url)
tensor_cache = await fetch_json(tensor_cache_url)

print('Metadata loaded.')

## Step 4: Start with the biggest numbers

This gives us a first summary of the model and its WebLLM package.

In [ ]:
original_parameters = original_info.get('safetensors', {}).get('total')
quantized_bytes = tensor_cache['metadata']['ParamBytes']
bits_per_param = tensor_cache['metadata']['BitsPerParam']
bf16_bytes = original_parameters * 2 if original_parameters else None
compression_ratio = (bf16_bytes / quantized_bytes) if bf16_bytes else None

summary_rows = [
    {'quantity': 'Original parameter count', 'value': format_int(original_parameters) if original_parameters else 'unknown'},
    {'quantity': 'That is about', 'value': format_billions(original_parameters) if original_parameters else 'unknown'},
    {'quantity': 'Quantized WebLLM weight bytes', 'value': format_int(quantized_bytes) + ' bytes'},
    {'quantity': 'Quantized WebLLM weight size', 'value': format_millions(quantized_bytes) + ' bytes'},
    {'quantity': 'Average bits per parameter', 'value': f"{bits_per_param:.2f} bits"},
    {'quantity': 'Approximate raw BF16 size', 'value': format_millions(bf16_bytes) + ' bytes' if bf16_bytes else 'unknown'},
    {'quantity': 'Approximate compression ratio', 'value': f"{compression_ratio:.2f}x smaller than BF16" if compression_ratio else 'unknown'},
]

pd.DataFrame(summary_rows)

## Step 5: What does a parameter mean?

A parameter is one learned number. But one number by itself is not very interesting. The real action comes from **large tables of numbers** that transform one vector into another.

The next cell uses a tiny toy example, not a real model weight file. It is just meant to show the basic idea of a weight matrix.

In [ ]:
toy_input = [1.0, 0.5, -0.5]
toy_weights = [
    [0.2, -0.4, 0.8],
    [1.1, 0.3, -0.2],
]


def matvec(matrix, vector):
    output = []
    for row in matrix:
        total = 0.0
        for weight, value in zip(row, vector):
            total += weight * value
        output.append(total)
    return output

toy_output = matvec(toy_weights, toy_input)
print('Input vector: ', toy_input)
print('Weight matrix:', toy_weights)
print('Output vector:', toy_output)

In plain language, each weight says how strongly one input number should push an output number up or down. A real language model just does this with much larger vectors and many more layers.

## Step 6: Read the model architecture

The WebLLM chat config tells us the broad shape of the network: how many layers it has, how wide its hidden state is, how many attention heads it uses, and how big its vocabulary is.

In [ ]:
text_config = chat_config['model_config']['text_config']
architecture_rows = [
    {'field': 'model_type', 'value': chat_config.get('model_type')},
    {'field': 'quantization', 'value': chat_config.get('quantization')},
    {'field': 'num_hidden_layers', 'value': text_config.get('num_hidden_layers')},
    {'field': 'hidden_size', 'value': text_config.get('hidden_size')},
    {'field': 'intermediate_size', 'value': text_config.get('intermediate_size')},
    {'field': 'num_attention_heads', 'value': text_config.get('num_attention_heads')},
    {'field': 'num_key_value_heads', 'value': text_config.get('num_key_value_heads')},
    {'field': 'head_dim', 'value': text_config.get('head_dim')},
    {'field': 'vocab_size', 'value': chat_config.get('vocab_size')},
    {'field': 'context_window_size', 'value': chat_config.get('context_window_size')},
]

pd.DataFrame(architecture_rows)

## Step 7: Flatten the tensor inventory

The converted WebLLM package stores weights in shard files such as `params_shard_0.bin`, `params_shard_1.bin`, and so on. Inside those files are many named tensors.

A **tensor** is just a container of numbers. A vector is a 1-dimensional tensor. A matrix is a 2-dimensional tensor.

In [ ]:
def flatten_tensor_records(cache):
    rows = []
    for shard in cache.get('records', []):
        shard_name = shard.get('dataPath')
        for record in shard.get('records', []):
            shape = record.get('shape', [])
            rows.append({
                'shard': shard_name,
                'name': record.get('name'),
                'shape': shape,
                'rank': len(shape),
                'dtype': record.get('dtype'),
                'nbytes': record.get('nbytes'),
            })
    return rows

tensor_rows = flatten_tensor_records(tensor_cache)
print('Tensors found:', len(tensor_rows))
print('Shard files found:', len(tensor_cache.get('records', [])))

## Step 8: How many matrices are in the package?

Most of the big learned transformations live in 2-dimensional tensors, which are matrices. Some other tensors are 1-dimensional vectors, such as normalization weights.

In [ ]:
matrix_count = sum(row['rank'] == 2 for row in tensor_rows)
vector_count = sum(row['rank'] == 1 for row in tensor_rows)
other_count = sum(row['rank'] not in (1, 2) for row in tensor_rows)

pd.DataFrame([
    {'kind': '2D tensors (matrices)', 'count': matrix_count},
    {'kind': '1D tensors (vectors)', 'count': vector_count},
    {'kind': 'Other tensor ranks', 'count': other_count},
    {'kind': 'Total tensors', 'count': len(tensor_rows)},
])

## Step 9: What kinds of tensors are these?

Tensor names tell us roughly what job they do. The next cell groups them into broad categories.

In [ ]:
def tensor_category(name):
    if 'embed_tokens' in name:
        return 'token embeddings'
    if 'self_attn' in name:
        return 'attention'
    if '.mlp.' in name:
        return 'feed-forward / MLP'
    if 'norm' in name:
        return 'normalization'
    if 'lm_head' in name:
        return 'output head'
    return 'other'

category_rows = []
for row in tensor_rows:
    category_rows.append(tensor_category(row['name']))

category_table = pd.Series(category_rows).value_counts().rename_axis('category').reset_index(name='tensor_count')
category_table

## Step 10: Look at a few real tensor names and shapes

These names may look intimidating at first, but they are useful clues:

- `embed_tokens` means the token embedding table
- `self_attn` means self-attention
- `mlp` means the feed-forward block
- `norm` means a normalization step
- the numbers like `layers.0` tell you which layer the tensor belongs to

In [ ]:
preview_table = pd.DataFrame(tensor_rows)[['name', 'shape', 'dtype', 'nbytes', 'shard']].head(20)
preview_table

## Step 11: What files are in the open-weight package?

When we say we downloaded the model, we did **not** download a magical black box. We downloaded a package of ordinary files. The converted WebLLM repository tells us what they are.

In [ ]:
file_rows = []
for item in converted_info.get('siblings', []):
    name = item.get('rfilename')
    if name.startswith('params_shard_') and name.endswith('.bin'):
        kind = 'weight shard'
    elif name == 'tensor-cache.json':
        kind = 'tensor inventory'
    elif name == 'mlc-chat-config.json':
        kind = 'runtime chat config'
    elif 'tokenizer' in name or 'added_tokens' in name:
        kind = 'tokenizer file'
    else:
        kind = 'other'
    file_rows.append({'file': name, 'kind': kind})

pd.DataFrame(file_rows)

## Step 12: What does "open weights" mean here?

For a beginner, here is a good plain-language definition:

- The **weights** are the learned numbers of the model.
- **Open weights** means those learned numbers have been published so other people can download and run them.
- That does **not** always mean every part of the training process is open. The training code, data, and license may be more or less open depending on the model family.

In this notebook, the original model repository holds the original published model weights, while the `mlc-ai` repository holds a converted and compressed package that WebLLM can run in the browser.

## Step 13: Why does the WebLLM package look different from the original model?

The WebLLM package has a different job. It is meant to run inside a browser, so it is usually:

- **quantized**, which means weights are stored with fewer bits
- **sharded**, which means split across several files
- paired with extra config files that tell WebLLM how to load and use the package

That is why you see files like `params_shard_3.bin` and `tensor-cache.json` instead of one giant `model.safetensors` file.

In [ ]:
original_files = [item.get('rfilename') for item in original_info.get('siblings', [])]
converted_files = [item.get('rfilename') for item in converted_info.get('siblings', [])]

comparison_rows = [
    {'package': 'Original model repo', 'file_count': len(original_files), 'examples': ', '.join(original_files[:6])},
    {'package': 'Converted WebLLM repo', 'file_count': len(converted_files), 'examples': ', '.join(converted_files[:6])},
]

pd.DataFrame(comparison_rows)

## Step 14: A freshman-level summary

If you had to explain this model package to a first-year college student, you could say:

1. A language model is mostly a huge pile of learned numbers.
2. Those numbers are stored in tensors, and many of the most important tensors are matrices.
3. The weights are "open" because people can download them.
4. WebLLM downloads a browser-ready version of those weights, compressed and split into shard files.
5. The tokenizer files teach the model how to break text into tokens.
6. The config files teach WebLLM how to load the package and chat with it.

So when we say "we downloaded a model," we really mean: we downloaded a carefully packaged collection of numbers plus the files needed to interpret them.

## Your turn

Change `MODEL_CHOICE` and compare two model families. Which one has more layers? Which one has a larger vocabulary? Which one needs more shard files?